# AG_PRAXIS NB03b — Timing Feature AblationThe provenance check in NB03 found that a forest can name which recording a row came from,and that the four timing features do it better on their own than all forty-four together.One of those four is not a measure of time. Duration holds the Time-To-Live header field,which is a small integer that a sending host sets and each hop decrements, so it describesthe path a packet took rather than the rate at which packets arrived.That matters here because a header field written by the sender is exactly the kind of columnthat can differ between one recording session and the next for reasons that have nothing todo with the attack. So the question this notebook asks is narrow: how much of NB03'scapture-identification result is Duration carrying?The way to find out is to run NB03 again with Duration taken out and read the two sets ofnumbers side by side. Everything else stays as it was — the same forest, the same fiftyrecordings, the same eight thousand rows drawn from each, the same seed — because a secondchange would make the difference unattributable.Four runs. The timing family without Duration, against NB03's 0.9301 for the family with it.All features without Duration, against 0.8010. The same again with the attack class heldfixed, against 0.8280. And Duration on its own, so the column is measured directly ratherthan only by its absence.This is a measurement and not a hypothesis test, and Duration stays in the forty-fourfeatures for every run already executed. Nothing here changes the feature set.

Cell one is the same setup every notebook in this project starts from. It mounts Drive so theartefacts survive the session, clones the repository so `src/` can be imported, and recordsthe commit the code came from, because a result without a commit cannot be reproduced.

In [ ]:
import osimport subprocessimport sysfrom datetime import datefrom pathlib import PathIN_COLAB = "google.colab" in sys.modulesREPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"if IN_COLAB:    from google.colab import drive    drive.mount("/content/drive")    REPO_ROOT = Path("/content/repo")    if REPO_ROOT.exists():        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)    else:        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)else:    REPO_ROOT = Path.cwd()    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:        REPO_ROOT = REPO_ROOT.parentif str(REPO_ROOT) not in sys.path:    sys.path.insert(0, str(REPO_ROOT))def git(*args):    return subprocess.run(        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True    ).stdout.strip()GIT_SHA = git("rev-parse", "--short", "HEAD")GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")GIT_DIRTY = bool(git("status", "--porcelain"))RUN_DATE = date.today().isoformat()print(f"colab     : {IN_COLAB}")print(f"repo root : {REPO_ROOT}")print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))print(f"run date  : {RUN_DATE}")

Next the configuration. Seed and paths come from `config/base.yaml` rather than being typedhere, so this notebook and NB03 cannot drift apart on either.There is a fast pass and a full pass. The fast pass draws a thousand rows from each recordinginstead of eight thousand and writes somewhere else, so it tests that the code runs withoutproducing anything that could be mistaken for a result. Only the full pass is comparable withNB03, because NB03's figures were measured on eight thousand.

In [ ]:
import jsonimport randomimport timeimport numpy as npimport pandas as pdimport yamlfrom src import captures as capfrom src import inventory as invfrom src.runs import assert_single_change, fit_and_saveCFG = inv.load_config(REPO_ROOT)SEED = CFG["seed"]TRAIN_DIR = Path(CFG["paths"]["train_dir"])TEST_DIR = Path(CFG["paths"]["test_dir"])ARTIFACTS = Path(CFG["paths"]["artifacts"])FAST = os.environ.get("FAST", "0") == "1"ROWS_PER_RECORDING = 1_000 if FAST else 8_000FAST_READ_ROWS = 20_000 if FAST else NoneOUT_DIR = ARTIFACTS / ("NB03b_fast" if FAST else "NB03b")OUT_DIR.mkdir(parents=True, exist_ok=True)if IN_COLAB and not ARTIFACTS.exists():    raise FileNotFoundError(        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "        "config/base.yaml is wrong. Nothing this notebook writes would survive."    )ALL_FILES = sorted(TRAIN_DIR.glob("*.csv")) + sorted(TEST_DIR.glob("*.csv"))if not ALL_FILES:    raise FileNotFoundError(f"no CSV files under {TRAIN_DIR} or {TEST_DIR}")pd.set_option("display.max_rows", 400)pd.set_option("display.width", 200)print(f"seed              : {SEED}")print(f"files             : {len(ALL_FILES)}")print(f"pass              : {'FAST, not a result' if FAST else 'FULL, the one the repository takes'}")print(f"rows per recording: {ROWS_PER_RECORDING:,}")print(f"artefacts         : {OUT_DIR}")

The column list comes from NB01's inventory rather than from reading a file here, so theforty-four features are the same forty-four every other notebook uses. Drate is the onecolumn constant across the whole dataset and is dropped, which is where forty-five becomesforty-four.

In [ ]:
INVENTORY_CANDIDATES = [    REPO_ROOT / "data" / "processed" / "dataset_inventory.json",    ARTIFACTS / "NB01" / "dataset_inventory.json",]INVENTORY_PATH = next((p for p in INVENTORY_CANDIDATES if p.exists()), None)if INVENTORY_PATH is None:    raise FileNotFoundError(        "dataset_inventory.json not found. NB01 has to have been run and its output moved "        f"into data/processed/. Looked in: {[str(p) for p in INVENTORY_CANDIDATES]}"    )INVENTORY = json.loads(INVENTORY_PATH.read_text())if INVENTORY.get("is_fast_pass"):    raise ValueError(f"{INVENTORY_PATH} was written by NB01's fast pass and is not a result")ALL_COLUMNS = list(INVENTORY["columns"])DROPPED = ["Drate"]FEATURES = [c for c in ALL_COLUMNS if c not in DROPPED]TIER_A = sorted(INVENTORY["tiers"]["A"])ROWS_PER_FILE = INVENTORY["rows_per_file"]print(f"inventory   : {INVENTORY_PATH}")print(f"  written by: {INVENTORY['generated_by']} at {INVENTORY['git_sha']}")print(f"columns     : {len(ALL_COLUMNS)}")print(f"dropped     : {DROPPED}")print(f"features    : {len(FEATURES)}")print(f"classes with several recordings: {len(TIER_A)}   {', '.join(TIER_A)}")assert len(FEATURES) == 44, f"expected 44 features after the drop, got {len(FEATURES)}"assert "Duration" in FEATURES, "Duration is not in the feature list, so there is nothing to ablate"

The figures this notebook is measured against are NB03's, and they are read out of the fileNB03 wrote rather than copied into this cell by hand. That way a mismatch between what NB03actually did and what this notebook assumes it did shows up as an error here instead of as awrong comparison at the end.The three asserts check the things that have to be the same for the comparison to meananything: the same forty-four features, the same fifty recordings, the same forest.

In [ ]:
VERDICT_CANDIDATES = [    REPO_ROOT / "data" / "processed" / "NB03_verdict.json",    ARTIFACTS / "NB03" / "NB03_verdict.json",]VERDICT_PATH = next((p for p in VERDICT_CANDIDATES if p.exists()), None)if VERDICT_PATH is None:    raise FileNotFoundError(        "NB03_verdict.json not found. NB03 has to have been run and its output moved into "        f"data/processed/. Looked in: {[str(p) for p in VERDICT_CANDIDATES]}"    )NB03 = json.loads(VERDICT_PATH.read_text())if NB03.get("is_fast_pass"):    raise ValueError(f"{VERDICT_PATH} was written by NB03's fast pass and is not a result")NB03_RUNS = {r["name"]: r for r in NB03["capture_identification"]}NB03_TIMING_FAMILY = list(NB03["families"]["timing"])NB03_FOREST = NB03["forest"]REFERENCE = {    "timing_family": NB03_RUNS["capture_family_timing"]["accuracy"],    "all_features": NB03_RUNS["capture_all_features"]["accuracy"],    "within_class_mean": NB03["within_class_mean_accuracy"],    "within_class_mean_chance": NB03["within_class_mean_chance"],    "timing_three": NB03_RUNS["capture_timing_three"]["accuracy"],}print(f"NB03 read from {VERDICT_PATH}")print(f"  run on          : {NB03['generated_on']} at {NB03['git_sha']}, seed {NB03['seed']}")print(f"  rows per record : {NB03['rows_per_recording']:,}")print(f"  timing family   : {', '.join(NB03_TIMING_FAMILY)}")print()print("the figures this notebook is measured against")print(f"  timing family, {len(NB03_TIMING_FAMILY)} features          : {REFERENCE['timing_family']:.4f}")print(f"  all {NB03['n_features']} features                     : {REFERENCE['all_features']:.4f}")print(f"  all {NB03['n_features']} features, attack held fixed  : {REFERENCE['within_class_mean']:.4f}"      f"   (mean chance {REFERENCE['within_class_mean_chance']:.4f})")print()print("also measured by NB03, and the same three columns run 1 uses")print(f"  IAT, Rate, Srate                : {REFERENCE['timing_three']:.4f}")assert sorted(NB03["features"]) == sorted(FEATURES), (    "NB03 ran on a different feature list from the one the inventory gives here")assert NB03["n_recordings"] == 50, f"NB03 used {NB03['n_recordings']} recordings, not 50"assert NB03["rows_per_recording"] == 8_000, "NB03 did not draw 8,000 rows per recording"assert NB03_FOREST["n_estimators"] == 50 and NB03_FOREST["min_samples_leaf"] == 100assert NB03_FOREST["test_fraction"] == 0.30 and NB03_FOREST["random_state"] == SEED

Now the four feature sets. The timing family is read from `config/feature_families.yaml`,which is where the project records what belongs to which family, and the ablated set is thatfamily with Duration taken out. The same subtraction gives the forty-three.One thing has to be said plainly before the runs. The three columns left in the timing familyafter Duration goes are IAT, Rate and Srate, and NB03 already ran a model on exactly thosethree, under the name `capture_timing_three`, scoring 0.893475. So run 1 is not a newmeasurement at all. It is a reproduction of that run, and it is here because if it does notland on the same number then nothing else in this notebook can be read as Duration's doing.The columns are given in the order that run used, IAT, Rate, Srate. The family file liststhem Rate, Srate, IAT. With three features a tree draws one column to consider at each split,so which column sits at which position is part of the run and not a presentation detail. Therun below takes NB03's order, and the check that follows it reports whether the two agreebefore any of the other three runs is read.

In [ ]:
FAMILIES_PATH = REPO_ROOT / "config" / "feature_families.yaml"FAMILY_DOC = yaml.safe_load(FAMILIES_PATH.read_text())TIMING = [c for c in FAMILY_DOC["families"]["timing"] if c in FEATURES]ABLATED = "Duration"# The order NB03's capture_timing_three run gave these three columns, which is not the# order the family file lists them in. Run 1 reproduces that run, so it takes that order.NB03_TIMING_THREE = ["IAT", "Rate", "Srate"]TIMING_NO_DURATION = list(NB03_TIMING_THREE)FEATURES_NO_DURATION = [c for c in FEATURES if c != ABLATED]DURATION_ONLY = [ABLATED]NB03_RUN_CONFIG = ARTIFACTS / "NB03" / "capture_timing_three" / "config.json"if NB03_RUN_CONFIG.exists():    recorded = list(json.loads(NB03_RUN_CONFIG.read_text())["features"])    assert recorded == TIMING_NO_DURATION, (        f"NB03's capture_timing_three ran on {recorded} and run 1 would use "        f"{TIMING_NO_DURATION}, so it would not be a reproduction of it"    )    ORDER_CHECKED = f"against {NB03_RUN_CONFIG}"else:    ORDER_CHECKED = "not available on this machine, order taken from the record in this cell"print(f"timing family                  {len(TIMING):>2}   {', '.join(TIMING)}   (family order)")print(f"timing family without Duration {len(TIMING_NO_DURATION):>2}   {', '.join(TIMING_NO_DURATION)}   (NB03's order)")print(f"all features                   {len(FEATURES):>2}")print(f"all features without Duration  {len(FEATURES_NO_DURATION):>2}")print(f"Duration alone                 {len(DURATION_ONLY):>2}   {', '.join(DURATION_ONLY)}")assert TIMING == NB03_TIMING_FAMILY, (    f"the timing family here is {TIMING} and NB03 ran on {NB03_TIMING_FAMILY}")assert len(TIMING_NO_DURATION) == 3, f"expected 3 timing features left, got {TIMING_NO_DURATION}"assert len(FEATURES_NO_DURATION) == 43, f"expected 43 features left, got {len(FEATURES_NO_DURATION)}"assert ABLATED not in FEATURES_NO_DURATION and ABLATED not in TIMING_NO_DURATIONassert sorted(TIMING_NO_DURATION) == sorted([c for c in TIMING if c != ABLATED]), (    "the three columns run 1 uses are not the timing family with Duration removed")print(f"order of the three checked     : {ORDER_CHECKED}")

The recordings are the same fifty NB03 used: the eight classes that were recorded more thanonce, one row per file, with the partition kept in the name because the two halves of asession are two separate recordings for this purpose.

In [ ]:
def recording_of(path):    """The recording a file holds: its name without the extension, partition included."""    meta = cap.parse_capture(Path(path).name)    return f"{meta['capture_id']}_{meta['partition']}"rows = []for path in ALL_FILES:    meta = cap.parse_capture(path.name)    if meta["label"] not in TIER_A:        continue    rows.append(        {            "recording": recording_of(path),            "label": meta["label"],            "partition": meta["partition"],            "rows_available": int(ROWS_PER_FILE[path.name]),            "path": str(path),        }    )RECORDINGS = pd.DataFrame(rows).sort_values(["label", "recording"]).reset_index(drop=True)assert pd.api.types.is_numeric_dtype(RECORDINGS["rows_available"]), (    f"rows_available is {RECORDINGS['rows_available'].dtype}, not numeric")assert RECORDINGS["recording"].is_unique, "two files claim the same recording identifier"LABEL_OF_RECORDING = dict(zip(RECORDINGS["recording"], RECORDINGS["label"]))RECORDINGS_PER_CLASS = {    label: RECORDINGS.loc[RECORDINGS["label"] == label, "recording"].tolist() for label in TIER_A}SMALLEST_RECORDING = int(RECORDINGS["rows_available"].min())print(f"recordings                    : {len(RECORDINGS)}")print(f"chance if naming the recording: {1 / len(RECORDINGS):.4f}")print(f"smallest recording            : {SMALLEST_RECORDING:,} rows")print()for label in TIER_A:    ids = RECORDINGS_PER_CLASS[label]    print(f"  {label:<12} {len(ids)} recordings, chance {1 / len(ids):.3f}")assert len(RECORDINGS) == 50, f"expected 50 recordings, got {len(RECORDINGS)}"assert sorted(RECORDINGS["recording"]) == sorted(NB03["recordings"]), (    "the recordings here are not the ones NB03 used")assert ROWS_PER_RECORDING <= SMALLEST_RECORDING, (    f"asking for {ROWS_PER_RECORDING:,} rows per recording and the smallest holds "    f"{SMALLEST_RECORDING:,}")

Seeding, before anything is built.

In [ ]:
random.seed(SEED)np.random.seed(SEED)print(f"seeded with {SEED}")

The rows are drawn the way NB03 drew them: a fixed number from each recording, so no recordingcan be identified by contributing more rows than the others, and the draw is made from agenerator seeded with the same seed in the same order. All forty-four columns are read, notforty-three, because run 4 needs Duration and because reading the same columns in the sameorder keeps this draw identical to NB03's.

In [ ]:
def load_rows():    rng = np.random.default_rng(SEED)    frames, taken = [], []    started = time.time()    for i, row in enumerate(RECORDINGS.itertuples(), start=1):        frame = pd.read_csv(row.path, usecols=FEATURES, nrows=FAST_READ_ROWS)        frame = frame[FEATURES]        wrong = {c: str(t) for c, t in frame.dtypes.items() if not pd.api.types.is_numeric_dtype(t)}        assert not wrong, f"{Path(row.path).name} has non-numeric feature columns: {wrong}"        available = len(frame)        if available < ROWS_PER_RECORDING:            raise ValueError(                f"{row.recording} offers {available:,} rows and {ROWS_PER_RECORDING:,} are "                "needed. Every recording has to contribute the same number of rows."            )        chosen = np.sort(rng.choice(available, size=ROWS_PER_RECORDING, replace=False))        part = frame.iloc[chosen].astype("float32").reset_index(drop=True)        part["recording"] = row.recording        part["label"] = row.label        frames.append(part)        taken.append({"recording": row.recording, "label": row.label, "rows_read": int(available)})        del frame        if i % 10 == 0 or i == len(RECORDINGS):            print(f"  {i:>2}/{len(RECORDINGS)} recordings, {time.time() - started:.0f}s")    data = pd.concat(frames, ignore_index=True)    del frames    drawn = pd.DataFrame(taken)    nonfinite = int((~np.isfinite(data[FEATURES].to_numpy(dtype=np.float32))).sum())    if nonfinite:        data[FEATURES] = data[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)    sizes = data.groupby("recording").size()    assert sizes.nunique() == 1, f"recordings contributed different row counts: {sizes.to_dict()}"    assert data["recording"].dtype == object, "the recording identifier is not a plain string"    assert data["label"].dtype == object, "the label is not a plain string"    return data, drawn, nonfinite, time.time() - startedDATA, DRAWN, NONFINITE, LOAD_S = load_rows()print()print(f"rows loaded               : {len(DATA):,}")print(f"rows read to get them     : {int(DRAWN['rows_read'].sum()):,}")print(f"rows per recording        : {ROWS_PER_RECORDING:,}, equal from all {len(RECORDINGS)}")print(f"values that are not finite: {NONFINITE:,}" + ("  replaced with zero" if NONFINITE else ""))print(f"loaded in                 : {LOAD_S:.0f}s")print()print("Duration, over the rows just loaded")print(f"  minimum {DATA['Duration'].min():,.0f}   maximum {DATA['Duration'].max():,.0f}   "      f"median {DATA['Duration'].median():,.0f}")print(f"  share at the modal value: {DATA['Duration'].value_counts(normalize=True).iloc[0]:.4f}")

One helper for the four runs, so that each of them differs from the others only in whichcolumns it is given. The forest is NB03's forest: fifty trees, a minimum of a hundred samplesin a leaf, thirty percent of the rows held out, seeded at 42. The target is the recording, notthe attack.`assert_single_change` runs at the top of every training cell below. Each run declares theNB03 run it descends from, and the check refuses to start if anything other than the featurelist differs from it.

In [ ]:
from sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import train_test_splitTEST_FRACTION = 0.30N_ESTIMATORS = 50MIN_SAMPLES_LEAF = 100TOP_FEATURES = 15def forest():    return RandomForestClassifier(        n_estimators=N_ESTIMATORS,        min_samples_leaf=MIN_SAMPLES_LEAF,        n_jobs=-1,        random_state=SEED,    )def spec(run_id, features, *, subset=None):    """The comparable part of a run: everything that defines it, feature list included."""    return {        "run_id": run_id,        "target": "recording",        "features": list(features),        "subset": subset,        "rows_per_recording": ROWS_PER_RECORDING,        "n_estimators": N_ESTIMATORS,        "min_samples_leaf": MIN_SAMPLES_LEAF,        "test_fraction": TEST_FRACTION,        "seed": SEED,    }def identify_recording(frame, *, name, features, notes, extra_config=None):    """Fit a forest whose target is the recording, and write the run, in one statement."""    target = frame["recording"].to_numpy(dtype=object).astype(str)    X = frame[features].to_numpy(dtype=np.float32)    X_train, X_test, y_train, y_test = train_test_split(        X, target, test_size=TEST_FRACTION, random_state=SEED, stratify=target    )    return fit_and_save(        OUT_DIR,        name,        forest(),        X_train,        y_train,        X_test,        y_test,        features=features,        target="recording",        notes=notes,        extra_config={            "ablated_column": ABLATED,            "test_fraction": TEST_FRACTION,            "n_recordings": int(len(set(target.tolist()))),            "rows_per_recording": ROWS_PER_RECORDING,            "is_fast_pass": bool(FAST),            **(extra_config or {}),        },        top_k=TOP_FEATURES,    )def line(run, *, scope, reference=None, note=""):    """One row of the comparison table: what was run, what it scored, and against what."""    m = run["metrics"]    return {        "name": run["name"],        "scope": scope,        "n_features": int(run["config"]["n_features"]),        "n_classes": int(m["n_classes"]),        "chance": float(m["chance_rate"]),        "accuracy": float(m["accuracy"]),        "macro_f1": float(m["macro_f1"]),        "nb03": None if reference is None else float(reference),        "difference": None if reference is None else float(m["accuracy"]) - float(reference),        "train_seconds": float(m["train_seconds"]),        "note": note,    }print(f"forest      : {N_ESTIMATORS} trees, min leaf {MIN_SAMPLES_LEAF}, "      f"{TEST_FRACTION:.0%} held out, seed {SEED}")print(f"target      : which of the {len(RECORDINGS)} recordings a row came from")print(f"column under test: {ABLATED}")

Run 1. The timing family with Duration removed, which leaves three columns. NB03 gave thewhole family 0.9301, and the parent this run declares is that family run, so the one thingthat differs between them is the column list. The same three columns in the same order arewhat NB03 ran as `capture_timing_three`, so this run has a second figure to answer to aswell, and the cell after next is where that is settled.

In [ ]:
PARENT_1 = spec("capture_family_timing", TIMING)CONFIG_1 = spec("timing_minus_duration", TIMING_NO_DURATION)print("one change from capture_family_timing:", sorted(assert_single_change(CONFIG_1, PARENT_1)))RUN_1 = identify_recording(    DATA,    name="timing_minus_duration",    features=TIMING_NO_DURATION,    notes="which recording a row came from, timing family with Duration removed",    extra_config={"parent": "capture_family_timing", "one_change": "features"},)

In [ ]:
LINE_1 = line(    RUN_1,    scope="timing family without Duration",    reference=REFERENCE["timing_family"],    note=", ".join(TIMING_NO_DURATION),)print(f"features    : {LINE_1['n_features']}   {LINE_1['note']}")print(f"recordings  : {LINE_1['n_classes']}   chance {LINE_1['chance']:.4f}")print(f"accuracy    : {LINE_1['accuracy']:.4f}")print(f"macro-F1    : {LINE_1['macro_f1']:.4f}")print(f"NB03, timing family with Duration : {LINE_1['nb03']:.4f}")print(f"difference                        : {LINE_1['difference']:+.4f}")print(f"NB03, IAT Rate Srate              : {REFERENCE['timing_three']:.4f}")print(f"difference against that run       : {LINE_1['accuracy'] - REFERENCE['timing_three']:+.4f}")print(f"trained in  : {RUN_1['metrics']['train_seconds']:.0f}s")print(f"written to  : {RUN_1['run_dir']}")

Before the other three runs are read, the reproduction has to be settled. Run 1 gave a forestthe same three columns in the same order as NB03's `capture_timing_three`, drawn from the samerows under the same seed and the same forest settings, so the two should return the samenumber.Not to the last bit, though. A forest at a fixed seed is reproducible within a session and notacross machines: a different BLAS build, a different thread count or a different scikit-learnversion can move the last few decimals, and NB03 ran in a different session from this one. Sothe check allows a difference of 0.0001, which is an order of magnitude below anything thatcould be read as a result and above the noise a change of environment produces. A differencelarger than that is the notebook moving for reasons of its own, and every comparison belowcarries at least that much movement whatever Duration does or does not contribute.

In [ ]:
REPRODUCTION_TOLERANCE = 1e-4RUN_1_DIFFERENCE = LINE_1["accuracy"] - REFERENCE["timing_three"]RUN_1_REPRODUCES = bool(abs(RUN_1_DIFFERENCE) <= REPRODUCTION_TOLERANCE)if RUN_1_REPRODUCES:    print("Run 1 reproduces NB03's capture_timing_three.")    print(f"  NB03      : {REFERENCE['timing_three']:.6f}")    print(f"  here      : {LINE_1['accuracy']:.6f}")    print(f"  difference: {RUN_1_DIFFERENCE:+.6f}, inside the {REPRODUCTION_TOLERANCE:g} allowed")    print("The three comparisons below are read against NB03 on that basis.")else:    print("!" * 79)    print("RUN 1 DID NOT REPRODUCE NB03's capture_timing_three")    print("!" * 79)    print(f"  NB03      : {REFERENCE['timing_three']:.6f}")    print(f"  here      : {LINE_1['accuracy']:.6f}")    print(f"  difference: {RUN_1_DIFFERENCE:+.6f}, against {REPRODUCTION_TOLERANCE:g} allowed")    print()    print("  The same three columns, the same order, the same rows, the same seed and the")    print("  same forest went into both, and the tolerance already allows for a change of")    print("  machine or library version, so this difference is the notebook moving and not")    print("  Duration's contribution. Read it before reading anything below: every")    print("  comparison here moves by at least this much for reasons that have nothing to")    print("  do with the column being removed.")    print("!" * 79)

Run 2. All the features except Duration, which leaves forty-three. NB03 gave the fullforty-four 0.8010.

In [ ]:
PARENT_2 = spec("capture_all_features", FEATURES)CONFIG_2 = spec("all_minus_duration", FEATURES_NO_DURATION)print("one change from capture_all_features:", sorted(assert_single_change(CONFIG_2, PARENT_2)))RUN_2 = identify_recording(    DATA,    name="all_minus_duration",    features=FEATURES_NO_DURATION,    notes="which recording a row came from, all features except Duration",    extra_config={"parent": "capture_all_features", "one_change": "features"},)

In [ ]:
LINE_2 = line(    RUN_2,    scope="all features without Duration",    reference=REFERENCE["all_features"],    note=f"{len(FEATURES_NO_DURATION)} of {len(FEATURES)} columns",)print(f"features    : {LINE_2['n_features']}")print(f"recordings  : {LINE_2['n_classes']}   chance {LINE_2['chance']:.4f}")print(f"accuracy    : {LINE_2['accuracy']:.4f}")print(f"macro-F1    : {LINE_2['macro_f1']:.4f}")print(f"NB03, all {len(FEATURES)} features        : {LINE_2['nb03']:.4f}")print(f"difference                    : {LINE_2['difference']:+.4f}")print(f"trained in  : {RUN_2['metrics']['train_seconds']:.0f}s")print()print(f"the {TOP_FEATURES} columns this forest relied on most, with Duration gone")top = pd.DataFrame(RUN_2["metrics"]["top_features"])top.insert(0, "rank", np.arange(1, len(top) + 1))print(top.assign(importance=lambda f: f["importance"].map(lambda v: f"{v:.4f}")).to_string(index=False))

Run 3 is the same forty-three columns with the attack class already known. One model perclass, so the only thing left to tell apart is one recording of a class from anotherrecording of the same class, and nothing the model does can be explained by it tellingattacks apart. NB03's figure for this is 0.8280, which is the mean over the eight classes.Eight fits rather than one, because holding the attack fixed means fitting inside each class.Each fit writes its own run, and the figure compared against NB03 is the mean of the eight,computed the same way NB03 computed it.

In [ ]:
RUNS_3, LINES_3 = {}, []for label in TIER_A:    parent = spec(f"capture_within_{label.replace('-', '_')}", FEATURES, subset=label)    config = spec(f"within_{label.replace('-', '_')}_minus_duration", FEATURES_NO_DURATION, subset=label)    assert_single_change(config, parent)    RUNS_3[label] = identify_recording(        DATA[DATA["label"] == label],        name=f"within_{label.replace('-', '_')}_minus_duration",        features=FEATURES_NO_DURATION,        notes=f"which {label} recording a row came from, all features except Duration",        extra_config={            "parent": f"capture_within_{label.replace('-', '_')}",            "one_change": "features",            "class": label,        },    )

In [ ]:
for label in TIER_A:    reference = NB03_RUNS[f"capture_within_{label.replace('-', '_')}"]["accuracy"]    LINES_3.append(        line(            RUNS_3[label],            scope=f"within {label}",            reference=reference,            note=f"{len(RECORDINGS_PER_CLASS[label])} recordings",        )    )TABLE_3 = pd.DataFrame(LINES_3)MEAN_3 = float(TABLE_3["accuracy"].mean())MEAN_3_CHANCE = float(TABLE_3["chance"].mean())print(    TABLE_3[["scope", "n_classes", "chance", "accuracy", "nb03", "difference"]]    .rename(columns={"n_classes": "recordings"})    .assign(        chance=lambda f: f["chance"].map(lambda v: f"{v:.4f}"),        accuracy=lambda f: f["accuracy"].map(lambda v: f"{v:.4f}"),        nb03=lambda f: f["nb03"].map(lambda v: f"{v:.4f}"),        difference=lambda f: f["difference"].map(lambda v: f"{v:+.4f}"),    )    .to_string(index=False))print()print(f"mean accuracy over the {len(TIER_A)} classes : {MEAN_3:.4f}")print(f"mean chance                          : {MEAN_3_CHANCE:.4f}")print(f"NB03, all {len(FEATURES)} features, attack fixed: {REFERENCE['within_class_mean']:.4f}")print(f"difference                           : {MEAN_3 - REFERENCE['within_class_mean']:+.4f}")assert abs(MEAN_3_CHANCE - REFERENCE["within_class_mean_chance"]) < 1e-9, (    "the mean chance rate here does not match NB03's, so the two means are over different sets")

Run 4 gives the model Duration and nothing else. The three runs above measure the column bywhat happens when it is taken away; this one measures it directly, against the same fiftyrecordings and the same chance rate of 0.02.

In [ ]:
PARENT_4 = spec("capture_family_timing", TIMING)CONFIG_4 = spec("duration_only", DURATION_ONLY)print("one change from capture_family_timing:", sorted(assert_single_change(CONFIG_4, PARENT_4)))RUN_4 = identify_recording(    DATA,    name="duration_only",    features=DURATION_ONLY,    notes="which recording a row came from, Duration alone",    extra_config={"parent": "capture_family_timing", "one_change": "features"},)

In [ ]:
LINE_4 = line(RUN_4, scope="Duration alone", note="single-feature reference")print(f"features    : {LINE_4['n_features']}   {ABLATED}")print(f"recordings  : {LINE_4['n_classes']}   chance {LINE_4['chance']:.4f}")print(f"accuracy    : {LINE_4['accuracy']:.4f}")print(f"macro-F1    : {LINE_4['macro_f1']:.4f}")print(f"times chance: {LINE_4['accuracy'] / LINE_4['chance']:,.1f}")print(f"trained in  : {RUN_4['metrics']['train_seconds']:.0f}s")print()print("This run has no NB03 counterpart, so there is nothing to difference it against. It is")print("reported on its own terms.")

The four results together, with NB03's figures beside them, and the same table written toDrive so the numbers can be read back without this notebook.

In [ ]:
LINE_3 = {    "name": "within_class_mean_minus_duration",    "scope": "all features without Duration, attack held fixed",    "n_features": len(FEATURES_NO_DURATION),    "n_classes": len(TIER_A),    "chance": MEAN_3_CHANCE,    "accuracy": MEAN_3,    "macro_f1": float(TABLE_3["macro_f1"].mean()),    "nb03": REFERENCE["within_class_mean"],    "difference": MEAN_3 - REFERENCE["within_class_mean"],    "train_seconds": float(TABLE_3["train_seconds"].sum()),    "note": f"mean over {len(TIER_A)} classes",}SUMMARY = pd.DataFrame([LINE_1, LINE_2, LINE_3, LINE_4])print(    SUMMARY[["scope", "n_features", "chance", "accuracy", "nb03", "difference"]]    .assign(        chance=lambda f: f["chance"].map(lambda v: f"{v:.4f}"),        accuracy=lambda f: f["accuracy"].map(lambda v: f"{v:.4f}"),        nb03=lambda f: f["nb03"].map(lambda v: "—" if pd.isna(v) else f"{v:.4f}"),        difference=lambda f: f["difference"].map(lambda v: "—" if pd.isna(v) else f"{v:+.4f}"),    )    .to_string(index=False))DOCUMENT = {    "generated_by": "AG_PRAXIS_NB03b_timing_ablation.ipynb",    "generated_on": RUN_DATE,    "git_sha": GIT_SHA,    "git_dirty": GIT_DIRTY,    "seed": SEED,    "is_fast_pass": bool(FAST),    "ablated_column": ABLATED,    "reference": {"source": str(VERDICT_PATH), **REFERENCE},    "n_recordings": int(len(RECORDINGS)),    "rows_per_recording": ROWS_PER_RECORDING,    "rows_loaded": int(len(DATA)),    "forest": {        "n_estimators": N_ESTIMATORS,        "min_samples_leaf": MIN_SAMPLES_LEAF,        "test_fraction": TEST_FRACTION,        "random_state": SEED,    },    "feature_sets": {        "timing_family": TIMING,        "timing_minus_duration": TIMING_NO_DURATION,        "all_minus_duration": FEATURES_NO_DURATION,        "duration_only": DURATION_ONLY,    },    "runs": SUMMARY.to_dict(orient="records"),    "within_class_runs": TABLE_3.to_dict(orient="records"),}SUMMARY_PATH = OUT_DIR / "duration_ablation.json"SUMMARY_PATH.write_text(json.dumps(cap.jsonable(DOCUMENT), indent=2, default=str) + "\n")print()print(f"wrote {SUMMARY_PATH}")print()print(f"the {2 + len(TIER_A) + 1} runs, each holding config.json, metrics.json, y_true.npy, "      "y_pred.npy and model.joblib")for name in ["timing_minus_duration", "all_minus_duration"] + [    f"within_{l.replace('-', '_')}_minus_duration" for l in TIER_A] + ["duration_only"]:    run_dir = OUT_DIR / name    metrics = json.loads((run_dir / "metrics.json").read_text())    size = sum(p.stat().st_size for p in run_dir.glob("*"))    print(f"  {name:<38} accuracy {metrics['accuracy']:.4f}   {size / 1e6:,.1f} MB")

The ledger entry, ready to paste into `RESULTS_LEDGER.md`.

In [ ]:
if FAST:    status = "DO NOT ENTER, fast pass"elif ROWS_PER_RECORDING != 8_000:    status = f"DO NOT ENTER, drew {ROWS_PER_RECORDING:,} rows per recording rather than 8,000"elif GIT_DIRTY:    status = "reported result, working tree dirty"else:    status = "reported result, not a hypothesis test"ledger = f"""### NB03b — timing feature ablation ({RUN_DATE})| field | value ||---|---|| notebook | AG_PRAXIS_NB03b_timing_ablation.ipynb || run date | {RUN_DATE} || git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} || seed | {SEED} || status | {status} || registered under | PREREGISTRATION.md Amendment 13 || column removed | {ABLATED}, the TTL header field || protocol | NB03's, unchanged: RandomForest, {N_ESTIMATORS} trees, min leaf {MIN_SAMPLES_LEAF}, {TEST_FRACTION:.0%} held out || recordings | {len(RECORDINGS)}, from the {len(TIER_A)} classes recorded more than once || rows used | {len(DATA):,}, {ROWS_PER_RECORDING:,} from every recording || NB03 figures read from | {VERDICT_PATH.name}, run {NB03['generated_on']} at {NB03['git_sha']} || timing family without Duration, {len(TIMING_NO_DURATION)} features (chance {LINE_1['chance']:.4f}) | {LINE_1['accuracy']:.4f} against NB03's {REFERENCE['timing_family']:.4f} with it, difference {LINE_1['difference']:+.4f} || all features without Duration, {len(FEATURES_NO_DURATION)} features | {LINE_2['accuracy']:.4f} against NB03's {REFERENCE['all_features']:.4f}, difference {LINE_2['difference']:+.4f} || the same, attack class held fixed | {MEAN_3:.4f} mean over {len(TIER_A)} classes against NB03's {REFERENCE['within_class_mean']:.4f}, difference {MEAN_3 - REFERENCE['within_class_mean']:+.4f} || Duration alone (chance {LINE_4['chance']:.4f}) | {LINE_4['accuracy']:.4f} accuracy, {LINE_4['macro_f1']:.4f} macro-F1 || run 1 reproduces NB03's capture_timing_three | {"yes, " + format(LINE_1['accuracy'], '.6f') + " against " + format(REFERENCE['timing_three'], '.6f') + ", difference " + format(RUN_1_DIFFERENCE, '+.6f') + ", inside the 0.0001 allowed" if RUN_1_REPRODUCES else "NO, " + format(LINE_1['accuracy'], '.6f') + " against " + format(REFERENCE['timing_three'], '.6f') + ", difference " + format(RUN_1_DIFFERENCE, '+.6f') + ", outside the 0.0001 allowed. Every comparison in this entry carries at least that much movement"} || runs written | {2 + len(TIER_A) + 1} || artefacts | {OUT_DIR}, holding duration_ablation.json and the run directories |Per class, attack held fixed, without Duration: {" · ".join(f"{r.scope.replace('within ', '')} {r.accuracy:.4f} (NB03 {r.nb03:.4f})" for r in TABLE_3.itertuples())}Duration stays in the 44 features for every run already executed. This notebook measures thecolumn's contribution to the NB03 figures and does not change the feature set."""print(ledger)